# 03-4. 반복문: for와 while 실습

이 노트북은 03-1의 자료형, 03-2의 컬렉션, 03-3의 조건식을 여러 데이터에 반복 적용합니다. 각 코드 셀은 **결과 예측 → 실행 → 결과 설명 → 입력 변경** 순서로 학습하세요. 함수와 예외 처리는 이후 절에서 다룹니다.

## 0. 학습 전 확인

아래 셀을 실행하기 전에 각 반복 횟수와 마지막 변수 값을 적어 보세요.

In [ ]:
for number in range(1, 5):
    print(number)

attempts = 0
while attempts < 3:
    attempts += 1

print("마지막 attempts:", attempts)
assert attempts == 3

## 1. iterable과 iterator

리스트는 반복 가능한 iterable입니다. `iter()`로 iterator를 만들고 `next()`로 값을 하나씩 소비해 `for`의 내부 동작을 관찰합니다.

In [ ]:
events = ["ALLOW", "DENY"]
event_iterator = iter(events)

first = next(event_iterator)
second = next(event_iterator)

print(first, second)
assert first == "ALLOW"
assert second == "DENY"

## 2. for, range, enumerate, zip

값만 필요하면 직접 순회하고, 정수 구간은 `range()`, 위치는 `enumerate()`, 대응 값은 `zip()`으로 얻습니다. `range()`의 끝값이 제외되는지 확인하세요.

In [ ]:
actions = ["ALLOW", "DENY"]

for action in actions:
    print("값:", action)

forward = list(range(1, 5))
backward = list(range(5, 0, -2))
print("정방향:", forward)
print("역방향:", backward)

assert forward == [1, 2, 3, 4]
assert backward == [5, 3, 1]

In [ ]:
ips = ["10.0.0.5", "198.51.100.9"]
actions = ["ALLOW", "DENY"]
rows = []

for line_number, (ip, action) in enumerate(
    zip(ips, actions, strict=True),
    start=1,
):
    row = f"{line_number}: {ip} -> {action}"
    rows.append(row)
    print(row)

assert rows == [
    "1: 10.0.0.5 -> ALLOW",
    "2: 198.51.100.9 -> DENY",
]

## 3. while의 종료 설계

`while`에서는 초기 상태, 계속 조건, 상태 갱신을 찾습니다. 다음 셀은 성공하거나 최대 세 번 시도하면 종료됩니다.

In [ ]:
responses = [False, False, True]
attempt = 0
is_connected = False

while attempt < 3 and not is_connected:
    is_connected = responses[attempt]
    attempt += 1
    print(f"시도 {attempt}: {is_connected}")

assert attempt == 3
assert is_connected is True

## 4. continue, break, 반복문의 else

빈 값은 `continue`로 건너뛰고 `STOP`은 `break`로 반복을 종료합니다. 반복문의 `else`는 `break` 없이 끝났을 때 실행됩니다.

In [ ]:
raw_actions = ["ALLOW", "", "DENY", "STOP", "DENY"]
processed_actions = []

for action in raw_actions:
    if action == "":
        continue
    if action == "STOP":
        break
    processed_actions.append(action)

print(processed_actions)
assert processed_actions == ["ALLOW", "DENY"]

In [ ]:
open_ports = [22, 80, 443]
target_port = 3389
target_found = False

for port in open_ports:
    if port == target_port:
        target_found = True
        break
else:
    print("대상 포트 없음")

assert target_found is False

## 5. 핵심 패턴: 카운트·누적·필터·변환·집계

반복 전에 결과 변수를 초기화하고 반복 안에서 갱신합니다. 각 결과의 자료형을 먼저 예상하세요.

In [ ]:
actions = ["ALLOW", "DENY", "DENY", "ALLOW"]
packet_sizes = [120, 80, 200]
raw_names = [" allow ", "Deny", "DENY"]
source_ips = ["10.0.0.5", "198.51.100.9", "198.51.100.9"]

deny_count = 0
denied_actions = []
for action in actions:
    if action == "DENY":
        deny_count += 1
        denied_actions.append(action)

total_size = 0
for size in packet_sizes:
    total_size += size

normalized_actions = []
for name in raw_names:
    normalized_actions.append(name.strip().upper())

count_by_ip = {}
unique_ips = set()
for ip in source_ips:
    count_by_ip[ip] = count_by_ip.get(ip, 0) + 1
    unique_ips.add(ip)

print(deny_count, total_size)
print(normalized_actions)
print(count_by_ip)
print(sorted(unique_ips))

assert deny_count == 2
assert denied_actions == ["DENY", "DENY"]
assert total_size == 400
assert normalized_actions == ["ALLOW", "DENY", "DENY"]
assert count_by_ip == {"10.0.0.5": 1, "198.51.100.9": 2}

## 6. 순회 중 자료구조를 안전하게 다루기

순회 중인 리스트나 딕셔너리의 크기를 직접 바꾸지 않습니다. 필요한 값으로 새 컬렉션을 만들거나 삭제 대상을 먼저 모읍니다.

In [ ]:
numbers = [1, 2, 2, 3]
filtered_numbers = []

for number in numbers:
    if number != 2:
        filtered_numbers.append(number)

counts = {"ALLOW": 0, "DENY": 2, "UNKNOWN": 0}
keys_to_delete = []

for key, count in counts.items():
    if count == 0:
        keys_to_delete.append(key)

for key in keys_to_delete:
    del counts[key]

print(filtered_numbers)
print(counts)
assert filtered_numbers == [1, 3]
assert counts == {"DENY": 2}

## 7. 중첩 반복, 집합 멤버십, 컴프리헨션

모든 조합이 필요한 경우에만 중첩 반복을 사용합니다. 단순한 멤버십 검사는 집합을 활용하고, 새 컬렉션을 만드는 짧은 변환·필터는 컴프리헨션으로 표현할 수 있습니다.

In [ ]:
hosts = ["web-1", "web-2"]
ports = [80, 443]
pairs = []

for host in hosts:
    for port in ports:
        pairs.append((host, port))

blocked_ips = {"198.51.100.9", "203.0.113.10"}
event_ips = ["10.0.0.5", "198.51.100.9", "192.0.2.7"]
matched_ips = [ip for ip in event_ips if ip in blocked_ips]

print(pairs)
print(matched_ips)
assert len(pairs) == len(hosts) * len(ports)
assert matched_ips == ["198.51.100.9"]

## 8. 미니 실습: 이벤트 목록 분석

유효성 검사, `continue`, 카운트, 리스트·집합·딕셔너리 누적을 결합합니다. 결과를 실행 전에 표로 예상하세요.

In [ ]:
events = [
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 443},
    {"action": "DENY", "ip": "198.51.100.9", "port": 22},
    {"action": "DENY", "ip": "198.51.100.9", "port": 3389},
    {"action": "BLOCK", "ip": "203.0.113.10", "port": 70000},
    {"action": "DENY", "ip": "203.0.113.10", "port": 443},
]

valid_actions = {"ALLOW", "DENY"}
sensitive_ports = {22, 3389}
valid_event_count = 0
invalid_events = []
denied_events = []
unique_ips = set()
deny_count_by_ip = {}
critical_events = []

for line_number, event in enumerate(events, start=1):
    action = event.get("action")
    port = event.get("port")
    has_valid_action = action in valid_actions
    has_valid_port = isinstance(port, int) and 1 <= port <= 65535

    if not (has_valid_action and has_valid_port):
        invalid_events.append({"line": line_number, "event": event})
        continue

    valid_event_count += 1
    unique_ips.add(event["ip"])

    if action == "DENY":
        denied_events.append(event)
        ip = event["ip"]
        deny_count_by_ip[ip] = deny_count_by_ip.get(ip, 0) + 1

        if port in sensitive_ports:
            critical_events.append(event)

summary = {
    "valid": valid_event_count,
    "invalid": len(invalid_events),
    "deny": len(denied_events),
    "unique_ip": len(unique_ips),
    "critical": len(critical_events),
}

print(summary)
print(deny_count_by_ip)

In [ ]:
expected_summary = {
    "valid": 4,
    "invalid": 1,
    "deny": 3,
    "unique_ip": 3,
    "critical": 2,
}

assert summary == expected_summary
assert deny_count_by_ip == {
    "198.51.100.9": 2,
    "203.0.113.10": 1,
}
assert invalid_events[0]["line"] == 4
print("모든 핵심 검증 통과")

## 9. 확장 실습

유효한 이벤트의 포트별 횟수를 집계하고 첫 번째 중요 이벤트를 찾습니다. 첫 번째 값을 찾는 즉시 `break`하고, 없을 때만 반복문의 `else`가 실행됩니다.

In [ ]:
count_by_port = {}

for event in events:
    action = event.get("action")
    port = event.get("port")
    is_valid = (
        action in valid_actions
        and isinstance(port, int)
        and 1 <= port <= 65535
    )
    if is_valid:
        count_by_port[port] = count_by_port.get(port, 0) + 1

first_critical_event = None
for event in events:
    if (
        event.get("action") == "DENY"
        and event.get("port") in sensitive_ports
    ):
        first_critical_event = event
        break
else:
    print("중요 이벤트 없음")

print(count_by_port)
print(first_critical_event)
assert count_by_port == {443: 2, 22: 1, 3389: 1}
assert first_critical_event == events[1]

## 10. 자기 점검

다음을 코드 수정과 말로 확인하세요.

1. `target_port`를 443으로 바꾸면 반복문의 `else`가 실행되지 않는 이유는 무엇인가요?
2. 미니 실습에 포트가 문자열인 이벤트를 추가하면 어느 목록에 들어가나요?
3. `valid_event_count = 0`을 반복문 안으로 옮기면 결과가 왜 잘못되나요?
4. `events`를 순회하면서 원본에서 잘못된 이벤트를 삭제하지 않은 이유는 무엇인가요?
5. IP별 횟수 대신 고유 IP만 필요하다면 어떤 자료구조가 가장 자연스러운가요?
6. 결과를 먼저 예측하고 입력의 첫 값·마지막 값·빈 값에서 `assert`로 검증해 보세요.